# Full linear penalty model

Defining full linear penalty model, in this case using 2 dwell positions and 3 dosing points for one structure in the ROI.

In [1]:
# Import matplotlib and numpy - helpful Python libraries for scientific computing
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from matplotlib import ticker
from math import pi
from decimal import Decimal

from scipy import optimize

In [2]:
# Define basic parameters
D_min = 2.73 # Gy
D_max = 10 # Gy
M_l = 1 # penalty/Gy
M_u = 1 # penalty/Gy

# Define dose rate matrix of size (r x c), where r = # dose points and c = # dwell positions
# Each cell Gy/s
# This would be read from a CSV file in actual planning; for now we will manually define the dose rates
# --> run MC sims to calculate dose rate contribution at a specified dose point from each dwell position + store in D matrix
D = np.array([[0.1, 0.05], [0.05, 0.05], [0.05, 0.1]])
num_dose_points = np.shape(D)[0]
num_dwell_pos = np.shape(D)[1]
print(D)

[[0.1  0.05]
 [0.05 0.05]
 [0.05 0.1 ]]


In [9]:
# Construct the bounds A_ub and b_ub
# Step 1 - construct A matrix for dwell time vector (size 2r x c)
A_t = np.vstack((-M_l*D[0,:], M_u*D[0,:]))
for j in range(1, num_dose_points):
    A_t = np.vstack((A_t, -M_l*D[j,:], M_u*D[j,:]))
print(A_t)

# Step 2 - construct A matrix for dose point vector (size 2r x r)
A_w = -1*np.identity(num_dose_points)
for j in range(0, num_dose_points):
    A_w = np.insert(A_w, 2*j, A_w[2*j,:], axis=0)
print(A_w)

# Step 3 - combine to create A_ub
A_ub = np.hstack((A_t, A_w))
print(A_ub)

# Step 4 - construct b_ub
b_ub = np.reshape(np.full((num_dose_points,2), [-M_l*D_min, M_u*D_max]), 2*num_dose_points)
print(b_ub)

[[-0.1  -0.05]
 [ 0.1   0.05]
 [-0.05 -0.05]
 [ 0.05  0.05]
 [-0.05 -0.1 ]
 [ 0.05  0.1 ]]
[[-1. -0. -0.]
 [-1. -0. -0.]
 [-0. -1. -0.]
 [-0. -1. -0.]
 [-0. -0. -1.]
 [-0. -0. -1.]]
[[-0.1  -0.05 -1.   -0.   -0.  ]
 [ 0.1   0.05 -1.   -0.   -0.  ]
 [-0.05 -0.05 -0.   -1.   -0.  ]
 [ 0.05  0.05 -0.   -1.   -0.  ]
 [-0.05 -0.1  -0.   -0.   -1.  ]
 [ 0.05  0.1  -0.   -0.   -1.  ]]
[-2.73 10.   -2.73 10.   -2.73 10.  ]


In [12]:
# Construct the parameter vector c
# Number of 0 entries = number of dwell positions
# Number of 1 entries = number of dose points
c = np.concatenate((np.zeros(num_dwell_pos), np.ones(num_dose_points)))
print(c)

[0. 0. 1. 1. 1.]


In [13]:
# Set up the optimization
from scipy.optimize import linprog
res = linprog(c, A_ub=A_ub, b_ub=b_ub)
print(res.fun)
print(res.x)
res.message

0.0
[-0.  54.6  0.   0.   0. ]


'Optimization terminated successfully. (HiGHS Status 7: Optimal)'